# Phase 4 — CIFAR-10 Training Experiments

Select a Python kernel with PyTorch, torchvision, and psutil, then choose **Run All**. The sweep trains seven models for 80 epochs in total. Each completed epoch is saved to `phase-4-metrics.csv`. At the end, the notebook displays the comparison and saves it as `phase-4-results.md` beside this notebook. Rerunning replaces those results.

The baseline is batch size 32, Adam, learning rate 0.001, and 10 epochs. Each run starts from the same seeded weights with a fresh optimizer. The data split, transforms, and validation batch size are fixed. With a constant learning rate, epochs 5, 10, and 20 of the same baseline run give the three epoch budgets. The CIFAR-10 test set is reserved for final evaluation after choosing a setting.

In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn
import torch
from copy import copy
from pathlib import Path
from time import perf_counter

import psutil  # If missing, run %pip install psutil in a notebook cell.
from torch.utils.data import Subset


In [2]:
project_dir = Path.cwd()
if (project_dir / 'cifar-10-from-scratch' / 'phase 4.ipynb').is_file():
    project_dir /= 'cifar-10-from-scratch'
data_dir = project_dir / 'data'

train_transform =  transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

training_dataset = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=True,
    transform=train_transform
)

indices = torch.randperm(
    len(training_dataset), generator=torch.Generator().manual_seed(42)
).tolist()
validation_dataset = copy(training_dataset)
validation_dataset.transform = transforms.ToTensor()
train_data = Subset(training_dataset, indices[5000:])
val_data = Subset(validation_dataset, indices[:5000])

### Building a CNN from scratch

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.testingStack = nn.Sequential(
            # 3x3 x 3 x 32 + 32
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            # 3x3 x 32 x 32 + 32
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 3x3 x 32 x 64 + 64
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 3x3 x 64x 128 + 128
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            # 128 x 10 + 10
            nn.Linear(128,10)
        )

    def forward(self,x):
        logits = self.testingStack(x)
        return logits
    
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

In [4]:
def train(trainloader, net, criterion, optimizer, sample_memory=None):
    net.train()
    total_loss = correct = total = 0

    for batch_index, (inputs, labels) in enumerate(trainloader):
        sample_batch = sample_memory is not None and batch_index % 10 == 0
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        if sample_batch:
            sample_memory()
        loss.backward()
        optimizer.step()
        if sample_batch:
            sample_memory()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.detach().argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

@torch.inference_mode()
def evaluate(loader, net, criterion):
    net.eval()
    total_loss = correct = total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = net(inputs)

        total_loss += criterion(outputs, labels).item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [5]:
import csv
from statistics import mean

from IPython.display import Markdown, display

PHASE4_SEED = 42
PHASE4_RUNS = [
    ('baseline', 32, 0.001, 'Adam', 20),
    ('batch-64', 64, 0.001, 'Adam', 10),
    ('batch-128', 128, 0.001, 'Adam', 10),
    ('batch-256', 256, 0.001, 'Adam', 10),
    ('lr-0.01', 32, 0.01, 'Adam', 10),
    ('lr-0.0001', 32, 0.0001, 'Adam', 10),
    ('sgd', 32, 0.001, 'SGD', 10),
]


def phase4_epochs(train_data, val_data, batch_size, lr, optimizer_name, epochs):
    torch.manual_seed(PHASE4_SEED)
    model = NeuralNetwork().to(device)
    optimizer = (torch.optim.Adam(model.parameters(), lr=lr) if optimizer_name == 'Adam'
                 else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9))
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(PHASE4_SEED))
    val_loader = DataLoader(val_data, batch_size=32)
    backend = torch.device(device).type
    process = psutil.Process()
    mib = 1024 ** 2

    for epoch in range(1, epochs + 1):
        rss_samples, mps_samples = [], []

        def sample_memory():
            rss_samples.append(process.memory_info().rss / mib)
            if backend == 'mps':
                mps_samples.append(torch.mps.current_allocated_memory() / mib)

        if backend == 'cuda':
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
        elif backend == 'mps':
            torch.mps.synchronize()
        start = perf_counter()
        train_loss, train_acc = train(loader, model, loss_fn, optimizer, sample_memory)
        val_loss, val_acc = evaluate(val_loader, model, loss_fn)
        if backend == 'cuda':
            torch.cuda.synchronize(device)
        elif backend == 'mps':
            torch.mps.synchronize()
        elapsed = perf_counter() - start
        yield dict(epoch=epoch, steps_per_epoch=len(loader), train_loss=train_loss,
                   val_loss=val_loss, train_acc=train_acc, val_acc=val_acc,
                   seconds=elapsed, rss_sampled_mib=max(rss_samples),
                   mps_sampled_mib=max(mps_samples, default=None),
                   cuda_peak_mib=(torch.cuda.max_memory_allocated(device) / mib
                                  if backend == 'cuda' else None))


def phase4_report(histories):
    lines = [
        '# Phase 4 — CIFAR-10 training experiments', '',
        f'Seed: {PHASE4_SEED}. Device: {device}. PyTorch: {torch.__version__}.',
        'One run per setting. Accuracy is shown as a percentage; CSV accuracy is a fraction.',
        'Epoch time includes training and validation. At a fixed epoch count, larger batches do fewer optimizer steps.',
        'RSS and MPS are the largest samples after forward and after the optimizer step on every tenth training batch, starting with the first. These are not true peaks. CUDA reports peak allocated tensor memory over the epoch. RSS is for the whole notebook process.',
    ]

    def add_table(title, headers, rows):
        lines.extend(['', f'## {title}', '', '| ' + ' | '.join(headers) + ' |',
                      '| ' + ' | '.join(['---'] * len(headers)) + ' |'])
        lines.extend('| ' + ' | '.join(map(str, row)) + ' |' for row in rows)

    def memory(rows, key):
        values = [row[key] for row in rows if row[key] is not None]
        return f'{max(values):.1f}' if values else '—'

    batch_rows = []
    for run_id in ('baseline', 'batch-64', 'batch-128', 'batch-256'):
        rows = histories[run_id][:10]
        last = rows[-1]
        batch_rows.append((last['batch_size'], last['steps_per_epoch'],
                           f"{rows[0]['train_loss']:.4f} → {last['train_loss']:.4f}",
                           f"{rows[0]['val_loss']:.4f} → {last['val_loss']:.4f}",
                           f"{last['train_acc']:.2%}", f"{last['val_acc']:.2%}",
                           f"{mean(r['seconds'] for r in rows):.2f}",
                           memory(rows, 'rss_sampled_mib'), memory(rows, 'mps_sampled_mib'),
                           memory(rows, 'cuda_peak_mib')))
    add_table('1. Batch size', ('Batch', 'Steps/epoch', 'Train loss (1 → 10)',
              'Val loss (1 → 10)', 'Train acc (10)', 'Val acc (10)', 'Mean s/epoch',
              'Sampled RSS MiB', 'Sampled MPS MiB', 'CUDA peak MiB'), batch_rows)

    def convergence(rows):
        first, mid, last = rows[0], rows[4], rows[-1]
        best = max(rows, key=lambda r: r['val_acc'])
        increases = sum(b['val_loss'] > a['val_loss'] for a, b in zip(rows, rows[1:]))
        return (f"{first['val_acc']:.2%}", f"{mid['val_acc']:.2%}", f"{last['val_acc']:.2%}",
                f"{best['val_acc']:.2%} (epoch {best['epoch']})",
                f"{first['val_loss']:.4f} → {last['val_loss']:.4f}", f'{increases}/{len(rows) - 1}')

    convergence_headers = ('Val acc (1)', 'Val acc (5)', 'Val acc (10)',
                           'Best val acc', 'Val loss (1 → 10)', 'Val loss increases')
    add_table('2. Learning rate', ('Learning rate', *convergence_headers),
              [(histories[key][0]['learning_rate'], *convergence(histories[key][:10]))
               for key in ('lr-0.01', 'baseline', 'lr-0.0001')])
    add_table('3. Optimizer', ('Optimizer', *convergence_headers),
              [(label, *convergence(histories[key][:10]))
               for key, label in [('baseline', 'Adam'), ('sgd', 'SGD (momentum 0.9)')]])
    lines.extend(['', 'Both optimizers use LR 0.001 here. Adam rescales updates using running gradient statistics, while momentum SGD accumulates gradients without that adaptive rescaling. The same LR can give very different effective updates, so tune LR (and momentum for SGD) separately before comparing their best performance.'])

    baseline = histories['baseline']
    epoch_rows = []
    previous_acc = None
    for cutoff in (5, 10, 20):
        rows = baseline[:cutoff]
        last = rows[-1]
        best = max(rows, key=lambda r: r['val_acc'])
        change = '—' if previous_acc is None else f"{100 * (last['val_acc'] - previous_acc):+.2f} pp"
        epoch_rows.append((cutoff, f"{last['train_acc']:.2%}", f"{last['val_acc']:.2%}",
                           change, f"{last['val_loss']:.4f}",
                           f"{best['val_acc']:.2%} (epoch {best['epoch']})",
                           f"{sum(r['seconds'] for r in rows):.1f}"))
        previous_acc = last['val_acc']
    add_table('4. Epoch budget', ('Epochs', 'Train acc', 'Val acc', 'Change in val acc',
              'Val loss', 'Best val acc so far', 'Total seconds'), epoch_rows)
    lines.extend(['', 'Small validation gains alongside falling training loss can indicate a plateau. Falling validation accuracy or rising validation loss while training improves can indicate overfitting. Check the full per-epoch history in `phase-4-metrics.csv`.', ''])
    return '\n'.join(lines)


def run_phase4(train_data, val_data, output_dir):
    metrics_path = output_dir / 'phase-4-metrics.csv'
    report_path = output_dir / 'phase-4-results.md'
    histories = {}
    with metrics_path.open('w', encoding='utf-8', newline='') as file:
        writer = None
        for run_id, batch_size, lr, optimizer_name, epochs in PHASE4_RUNS:
            histories[run_id] = []
            print(f'\n{run_id}: batch={batch_size}, LR={lr:g}, {optimizer_name}, {epochs} epochs')
            for metrics in phase4_epochs(train_data, val_data, batch_size, lr, optimizer_name, epochs):
                row = dict(run=run_id, batch_size=batch_size, learning_rate=lr, optimizer=optimizer_name,
                           momentum=0.9 if optimizer_name == 'SGD' else None,
                           seed=PHASE4_SEED, device=device, **metrics)
                if writer is None:
                    writer = csv.DictWriter(file, fieldnames=list(row))
                    writer.writeheader()
                writer.writerow(row)
                file.flush()
                histories[run_id].append(row)
                print(f"  {row['epoch']:2d}/{epochs} | train: {row['train_loss']:.4f}, {row['train_acc']:.2%}"
                      f" | val: {row['val_loss']:.4f}, {row['val_acc']:.2%} | {row['seconds']:.2f}s")
    report = phase4_report(histories)
    report_path.write_text(report, encoding='utf-8')
    print(f'\nSaved {metrics_path} and {report_path}')
    display(Markdown(report))
    return histories

In [6]:
phase4_histories = run_phase4(train_data, val_data, project_dir) 


baseline: batch=32, LR=0.001, Adam, 20 epochs
   1/20 | train: 1.8464, 29.73% | val: 1.6071, 40.56% | 13.93s
   2/20 | train: 1.4645, 46.12% | val: 1.3120, 52.10% | 12.77s
   3/20 | train: 1.2617, 54.23% | val: 1.1765, 58.38% | 13.08s
   4/20 | train: 1.1297, 59.80% | val: 1.0328, 63.62% | 12.49s
   5/20 | train: 1.0364, 63.40% | val: 0.9862, 65.06% | 12.07s
   6/20 | train: 0.9710, 65.57% | val: 0.9212, 68.26% | 11.82s
   7/20 | train: 0.9168, 67.79% | val: 0.8527, 70.58% | 12.65s
   8/20 | train: 0.8697, 69.48% | val: 0.8176, 71.36% | 12.20s
   9/20 | train: 0.8236, 71.04% | val: 0.8150, 71.50% | 12.40s
  10/20 | train: 0.7915, 72.49% | val: 0.8327, 71.24% | 12.29s
  11/20 | train: 0.7632, 73.52% | val: 0.7569, 73.86% | 12.26s
  12/20 | train: 0.7340, 74.50% | val: 0.7319, 75.34% | 12.28s
  13/20 | train: 0.7075, 75.42% | val: 0.7439, 75.34% | 12.26s
  14/20 | train: 0.6819, 76.26% | val: 0.6822, 76.30% | 12.30s
  15/20 | train: 0.6654, 76.95% | val: 0.6656, 77.30% | 12.29s
  16/20 

# Phase 4 — CIFAR-10 training experiments

Seed: 42. Device: mps. PyTorch: 2.15.0.
One run per setting. Accuracy is shown as a percentage; CSV accuracy is a fraction.
Epoch time includes training and validation. At a fixed epoch count, larger batches do fewer optimizer steps.
RSS and MPS are the largest samples after forward and after the optimizer step on every tenth training batch, starting with the first. These are not true peaks. CUDA reports peak allocated tensor memory over the epoch. RSS is for the whole notebook process.

## 1. Batch size

| Batch | Steps/epoch | Train loss (1 → 10) | Val loss (1 → 10) | Train acc (10) | Val acc (10) | Mean s/epoch | Sampled RSS MiB | Sampled MPS MiB | CUDA peak MiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 32 | 1407 | 1.8464 → 0.7915 | 1.6071 → 0.8327 | 72.49% | 71.24% | 12.57 | 801.2 | 14.1 | — |
| 64 | 704 | 1.8999 → 0.9484 | 1.6681 → 0.9159 | 66.72% | 68.18% | 8.58 | 467.7 | 27.0 | — |
| 128 | 352 | 1.9828 → 1.0692 | 1.7942 → 1.0168 | 62.28% | 64.08% | 7.38 | 498.3 | 53.8 | — |
| 256 | 176 | 2.0512 → 1.2631 | 1.8607 → 1.2167 | 54.49% | 56.00% | 7.19 | 560.9 | 104.3 | — |

## 2. Learning rate

| Learning rate | Val acc (1) | Val acc (5) | Val acc (10) | Best val acc | Val loss (1 → 10) | Val loss increases |
| --- | --- | --- | --- | --- | --- | --- |
| 0.01 | 9.94% | 10.60% | 10.34% | 10.60% (epoch 5) | 2.3031 → 2.3039 | 5/9 |
| 0.001 | 40.56% | 65.06% | 71.24% | 71.50% (epoch 9) | 1.6071 → 0.8327 | 1/9 |
| 0.0001 | 28.98% | 39.24% | 41.48% | 41.48% (epoch 10) | 1.9222 → 1.6044 | 2/9 |

## 3. Optimizer

| Optimizer | Val acc (1) | Val acc (5) | Val acc (10) | Best val acc | Val loss (1 → 10) | Val loss increases |
| --- | --- | --- | --- | --- | --- | --- |
| Adam | 40.56% | 65.06% | 71.24% | 71.50% (epoch 9) | 1.6071 → 0.8327 | 1/9 |
| SGD (momentum 0.9) | 15.36% | 28.34% | 35.12% | 35.86% (epoch 9) | 2.3004 → 1.7415 | 1/9 |

Both optimizers use LR 0.001 here. Adam rescales updates using running gradient statistics, while momentum SGD accumulates gradients without that adaptive rescaling. The same LR can give very different effective updates, so tune LR (and momentum for SGD) separately before comparing their best performance.

## 4. Epoch budget

| Epochs | Train acc | Val acc | Change in val acc | Val loss | Best val acc so far | Total seconds |
| --- | --- | --- | --- | --- | --- | --- |
| 5 | 63.40% | 65.06% | — | 0.9862 | 65.06% (epoch 5) | 64.3 |
| 10 | 72.49% | 71.24% | +6.18 pp | 0.8327 | 71.50% (epoch 9) | 125.7 |
| 20 | 79.62% | 77.12% | +5.88 pp | 0.7105 | 78.32% (epoch 19) | 239.9 |

Small validation gains alongside falling training loss can indicate a plateau. Falling validation accuracy or rising validation loss while training improves can indicate overfitting. Check the full per-epoch history in `phase-4-metrics.csv`.
